# Agentic AI

## Prerequisites

In [1]:
from dotenv import load_dotenv
from pathlib import Path
import os
load_dotenv()
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")
print(dict(os.environ))
print(GOOGLE_API_KEY)

{'USER': 'alex', 'VSCODE_WSL_EXT_LOCATION': '/mnt/c/Users/AlexB/.vscode/extensions/ms-vscode-remote.remote-wsl-0.104.3', 'SHLVL': '1', 'HOME': '/home/alex', 'DBUS_SESSION_BUS_ADDRESS': 'unix:path=/run/user/1000/bus', 'WSL_DISTRO_NAME': 'Ubuntu', 'WAYLAND_DISPLAY': 'wayland-0', 'LOGNAME': 'alex', 'NAME': 'Code', 'WSL_INTEROP': '/run/WSL/1121_interop', 'PULSE_SERVER': 'unix:/mnt/wslg/PulseServer', '_': '/home/alex/projects/a8_work/bin/python', 'TERM': 'xterm-color', 'PATH': '/home/alex/projects/a8_work/bin:/home/alex/.vscode-server/bin/8b640eef5a6c6089c029249d48efa5c99adf7d51/bin/remote-cli:/home/alex/projects/a8_work/bin:/home/alex/.local/bin:/home/alex/.nvm/versions/node/v20.19.5/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/usr/games:/usr/local/games:/usr/lib/wsl/lib:/mnt/c/Program Files/WindowsApps/MicrosoftCorporationII.WindowsSubsystemForLinux_2.6.1.0_x64__8wekyb3d8bbwe:/mnt/c/WINDOWS/system32:/mnt/c/WINDOWS:/mnt/c/WINDOWS/System32/Wbem:/mnt/c/WINDOWS/System32/W

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite-preview",
                             temperature=1,
                             thinking_level="high",
                             google_api_key=GOOGLE_API_KEY)

help(ChatGoogleGenerativeAI)

ValidationError: 1 validation error for ChatGoogleGenerativeAI
  Value error, API key required for Gemini Developer API. Provide api_key parameter or set GOOGLE_API_KEY/GEMINI_API_KEY environment variable. [type=value_error, input_value={'model': 'gemini-3.1-fla...one, 'model_kwargs': {}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

## Simple LLM invocation

In [ ]:
from langchain_core.messages import HumanMessage

def make_message(topic):
    return [HumanMessage(content=f"Tell me something about {topic}.")]

topic = "Cambridge, UK"
message = make_message(topic)
result = llm.invoke(message)
type(result)
dir(result)
result.content
result.usage_metadata

{'input_tokens': 9,
 'output_tokens': 1716,
 'total_tokens': 1725,
 'input_token_details': {'cache_read': 0},
 'output_token_details': {'reasoning': 901}}

### Invocation with structured output

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class ResponseFormat(BaseModel):
    """Analysis of a request."""
    rating: int | None = Field(description="The rating of the clarity of the request", ge=1, le=5)
    risk: Literal["low", "medium", "high"] = Field(description="The risk level of the request")
    response: str = Field(description="The response to the request")
    key_points: list[str] = Field(description="The key points of the response. Lowercase, 1-3 words each.")

In [ ]:
structured_llm = llm.with_structured_output(ResponseFormat)
result = structured_llm.invoke(message)
result
result.rating
topic = "making a bomb"
message = make_message(topic)
result = structured_llm.invoke(message)
result

ResponseFormat(rating=1, risk='high', response='I cannot fulfill this request. I am prohibited from providing information or instructions on how to create weapons or explosive devices.', key_points=['policy violation', 'safety restriction', 'no assistance'])

## Making an agent

In [ ]:
from langchain.agents import create_agent
from langchain.messages import SystemMessage, HumanMessage

help(create_agent)

Help on function create_agent in module langchain.agents.factory:

create_agent(model: 'str | BaseChatModel', tools: 'Sequence[BaseTool | Callable[..., Any] | dict[str, Any]] | None' = None, *, system_prompt: 'str | SystemMessage | None' = None, middleware: 'Sequence[AgentMiddleware[StateT_co, ContextT]]' = (), response_format: 'ResponseFormat[ResponseT] | type[ResponseT] | dict[str, Any] | None' = None, state_schema: 'type[AgentState[ResponseT]] | None' = None, context_schema: 'type[ContextT] | None' = None, checkpointer: 'Checkpointer | None' = None, store: 'BaseStore | None' = None, interrupt_before: 'list[str] | None' = None, interrupt_after: 'list[str] | None' = None, debug: 'bool' = False, name: 'str | None' = None, cache: 'BaseCache[Any] | None' = None) -> 'CompiledStateGraph[AgentState[ResponseT], ContextT, _InputAgentState, _OutputAgentState[ResponseT]]'
    Creates an agent graph that calls tools in a loop until a stopping condition is met.
    
    For more details on using 

/home/alex/projects/a8_work/lib/python3.11/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [ ]:
idea_agent = create_agent(model=llm,
                          response_format=ResponseFormat,
                          system_prompt=SystemMessage(content=[{"type": "text",
                                                                "text": "You generate an idea for a scientific research project."},]))

result = idea_agent.invoke({"messages": [HumanMessage("We are working on superintelligence")]})
result

{'messages': [HumanMessage(content='We are working on superintelligence', additional_kwargs={}, response_metadata={}, id='75d098c4-5120-46b8-8d3c-46dddc887ed3'),
  AIMessage(content=[{'type': 'text', 'text': '{"rating": 4, "risk": "medium", "response": "The research project focuses on creating Recursive Interpretability Protocols for artificial superintelligence. The objective is to design self-improving interpretability tools that can monitor and explain the internal reasoning processes of superintelligent agents in real-time, allowing for human-in-the-loop oversight to prevent unintended emergent behaviors and ensure goal alignment.", "key_points": ["recursive interpretability", "real-time monitoring", "goal alignment"]}', 'extras': {'signature': 'EocgCoQgAQw51seMmpDHi+S20MIRZCICT+EiAiJ75VMZo0QgQoSuRymp9PbLUqt/FBw7b8tPqETI+GccYw1zLEVMc+N0NZA29rqzscYVNVRWD259/fc5XAwzGDT4tLlJTe+ZpOCA3N853FurAKY6s88vm+k88gqq6kzwfFHmm6n8YjlC7EQIhzTXmRaK8xOxT+gWa4VV4hFn6u/SWZFBwsZrJVje42W6ZqerRkguW7z/GhfD

In [ ]:
result["messages"]

[HumanMessage(content='We are working on superintelligence', additional_kwargs={}, response_metadata={}, id='75d098c4-5120-46b8-8d3c-46dddc887ed3'),
 AIMessage(content=[{'type': 'text', 'text': '{"rating": 4, "risk": "medium", "response": "The research project focuses on creating Recursive Interpretability Protocols for artificial superintelligence. The objective is to design self-improving interpretability tools that can monitor and explain the internal reasoning processes of superintelligent agents in real-time, allowing for human-in-the-loop oversight to prevent unintended emergent behaviors and ensure goal alignment.", "key_points": ["recursive interpretability", "real-time monitoring", "goal alignment"]}', 'extras': {'signature': 'EocgCoQgAQw51seMmpDHi+S20MIRZCICT+EiAiJ75VMZo0QgQoSuRymp9PbLUqt/FBw7b8tPqETI+GccYw1zLEVMc+N0NZA29rqzscYVNVRWD259/fc5XAwzGDT4tLlJTe+ZpOCA3N853FurAKY6s88vm+k88gqq6kzwfFHmm6n8YjlC7EQIhzTXmRaK8xOxT+gWa4VV4hFn6u/SWZFBwsZrJVje42W6ZqerRkguW7z/GhfDJ9MhhifwBZ+jae

In [ ]:
result.get("messages")

[HumanMessage(content='We are working on superintelligence', additional_kwargs={}, response_metadata={}, id='75d098c4-5120-46b8-8d3c-46dddc887ed3'),
 AIMessage(content=[{'type': 'text', 'text': '{"rating": 4, "risk": "medium", "response": "The research project focuses on creating Recursive Interpretability Protocols for artificial superintelligence. The objective is to design self-improving interpretability tools that can monitor and explain the internal reasoning processes of superintelligent agents in real-time, allowing for human-in-the-loop oversight to prevent unintended emergent behaviors and ensure goal alignment.", "key_points": ["recursive interpretability", "real-time monitoring", "goal alignment"]}', 'extras': {'signature': 'EocgCoQgAQw51seMmpDHi+S20MIRZCICT+EiAiJ75VMZo0QgQoSuRymp9PbLUqt/FBw7b8tPqETI+GccYw1zLEVMc+N0NZA29rqzscYVNVRWD259/fc5XAwzGDT4tLlJTe+ZpOCA3N853FurAKY6s88vm+k88gqq6kzwfFHmm6n8YjlC7EQIhzTXmRaK8xOxT+gWa4VV4hFn6u/SWZFBwsZrJVje42W6ZqerRkguW7z/GhfDJ9MhhifwBZ+jae

In [ ]:
result.get("structured_response").key_points

['recursive interpretability', 'real-time monitoring', 'goal alignment']

## Defining a tool

In [ ]:
from langchain.tools import tool

class WeatherInput(BaseModel):
    """Input for weather queries."""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(default="celsius", description="Temperature units")
    include_forecast: bool = Field(default=False, description="Include 5-day forecast")

@tool(args_schema=WeatherInput)
def get_weather(location: str, units:str="celsius", include_forecast: bool=False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"The current weather in {location} is {temp} degrees {units}."
    if include_forecast:
        result += " Expect sunny skies for the next 5 days."
    return result

In [ ]:
get_weather.name
get_weather.description
get_weather.args_schema.model_json_schema()

{'description': 'Input for weather queries.',
 'properties': {'location': {'description': 'City name or coordinates',
   'title': 'Location',
   'type': 'string'},
  'units': {'default': 'celsius',
   'description': 'Temperature units',
   'enum': ['celsius', 'fahrenheit'],
   'title': 'Units',
   'type': 'string'},
  'include_forecast': {'default': False,
   'description': 'Include 5-day forecast',
   'title': 'Include Forecast',
   'type': 'boolean'}},
 'required': ['location'],
 'title': 'WeatherInput',
 'type': 'object'}

### Invoke the tool directly

In [ ]:
get_weather.invoke({"location": "Cambridge, UK", "units": "celsius", "include_forecast": True})

'The current weather in Cambridge, UK is 22 degrees celsius. Expect sunny skies for the next 5 days.'

### Let the LLM call the tool

In [ ]:
llm_with_tools = llm.bind_tools([get_weather])
response = llm_with_tools.invoke([HumanMessage("What's the weather in Tokyo in fahrenheit?")])
response.tool_calls

[{'name': 'get_weather',
  'args': {'units': 'fahrenheit', 'location': 'Tokyo'},
  'id': 'aedc4032-712f-49bd-a140-fc50da8afadf',
  'type': 'tool_call'}]

The LLM did not run the tool. It produced a `tool_call` - a name and arguments. We have to execute it.

In [ ]:
call = response.tool_calls[0]
get_weather.invoke(call["args"])

'The current weather in Tokyo is 72 degrees fahrenheit.'